# 05 · Conditional edges (routing)

**Goal:** by the end you can send the state to different nodes depending on its values, using a router function and
`add_conditional_edges`, and draw a diagram that shows every possible route.

## The idea in plain English

So far every edge was fixed: after `parse`, always `calculate`. A **conditional edge** chooses the next node while the
graph runs. After a node finishes, LangGraph calls a **router function**. The router looks at the state and returns
the *name* of the next node. It doesn't change the state; it only decides where to go.

We use this to give each operation its own node: `add`, `subtract`, `multiply`, `divide`. Unknown operators, like
`%`, go to an `error` node. In lesson 04 these choices were hidden in an `if/elif` inside `calculate`. Now they are
part of the graph, so the diagram shows every route.

**Analogy:** a mail sorter. The sorter reads the address on each letter (the state) and drops it into the right
chute (the next node). It never changes the letter.

## Picture

Dashed arrows are conditional: only one of them is taken on each run.

```mermaid
flowchart LR
    S([START]) --> parse
    parse -.-> add
    parse -.-> subtract
    parse -.-> multiply
    parse -.-> divide
    parse -.-> error
    add --> format
    subtract --> format
    multiply --> format
    divide --> format
    format --> E([END])
    error --> E
```

## Step 1: State and the `parse` node

The same state and `parse` node as the lesson 04 pipeline.

In [1]:
from typing import Literal, TypedDict

from langgraph.graph import StateGraph, START, END


class CalcState(TypedDict):
    expression: str
    a: float
    op: str
    b: float
    result: float
    answer: str


def parse(state: CalcState) -> dict:
    left, op, right = state["expression"].split()
    return {"a": float(left), "op": op, "b": float(right)}

## Step 2: One node per operation

Each operation is now a tiny node of its own. Each one also prints its name, so the output shows which route the
state took. (Lesson 15 shows a cleaner way to watch a graph run, called streaming.)

In [2]:
def add(state: CalcState) -> dict:
    print("  -> add")
    return {"result": state["a"] + state["b"]}


def subtract(state: CalcState) -> dict:
    print("  -> subtract")
    return {"result": state["a"] - state["b"]}


def multiply(state: CalcState) -> dict:
    print("  -> multiply")
    return {"result": state["a"] * state["b"]}


def divide(state: CalcState) -> dict:
    print("  -> divide")
    return {"result": state["a"] / state["b"]}

## Step 3: The `format` and `error` nodes

`format` builds the answer for a successful calculation. `error` builds a friendly message for an operator we don't
know. It never sets `result`.

In [3]:
def format_answer(state: CalcState) -> dict:
    return {"answer": f"{state['expression']} = {state['result']:g}"}


def error(state: CalcState) -> dict:
    print("  -> error")
    return {"answer": f"Sorry, I don't know the operator {state['op']!r}. Use +, -, * or /."}

## Step 4: The router function

`route` reads `state["op"]` and returns the name of the next node. `OPERATIONS.get(op, "error")` looks the symbol
up in the dict and falls back to `"error"` for anything not in it.

The return type `Literal["add", "subtract", "multiply", "divide", "error"]` is a type hint meaning "one of exactly
these strings". It is important: LangGraph reads it to learn every node the router might pick. That is how it
checks the names and draws the dashed arrows.

In [4]:
OPERATIONS = {"+": "add", "-": "subtract", "*": "multiply", "/": "divide"}


def route(state: CalcState) -> Literal["add", "subtract", "multiply", "divide", "error"]:
    return OPERATIONS.get(state["op"], "error")

A router is also just a function, so we can test it directly before putting it in a graph.

In [5]:
for op in ["+", "-", "*", "/", "%"]:
    print(f"{op!r} routes to {route({'op': op})!r}")

'+' routes to 'add'
'-' routes to 'subtract'
'*' routes to 'multiply'
'/' routes to 'divide'
'%' routes to 'error'


## Step 5: Build the graph with `add_conditional_edges`

`add_conditional_edges("parse", route)` means "when `parse` finishes, call `route` and go to the node it names".
It replaces the fixed `add_edge("parse", ...)`. The four operation nodes all lead to `format`. The `error` node goes
straight to `END`, because there is no result to format.

In [6]:
builder = StateGraph(CalcState)
builder.add_node("parse", parse)
for name, fn in [("add", add), ("subtract", subtract), ("multiply", multiply), ("divide", divide)]:
    builder.add_node(name, fn)
    builder.add_edge(name, "format")
builder.add_node("format", format_answer)
builder.add_node("error", error)

builder.add_edge(START, "parse")
builder.add_conditional_edges("parse", route)
builder.add_edge("format", END)
builder.add_edge("error", END)

graph = builder.compile()

## Step 6: Draw it

In Mermaid, `-.->` is a dashed (conditional) arrow and `-->` is a solid (fixed) one. You should see dashed arrows
from `parse` to all five nodes, taken from the `Literal` hint.

In [7]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	add(add)
	subtract(subtract)
	multiply(multiply)
	divide(divide)
	format(format)
	error(error)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	add --> format;
	divide --> format;
	multiply --> format;
	parse -.-> add;
	parse -.-> divide;
	parse -.-> error;
	parse -.-> multiply;
	parse -.-> subtract;
	subtract --> format;
	error --> __end__;
	format --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 7: Invoke with different operators

Each run takes exactly one route. The `-> name` lines are printed by the node that ran.

In [8]:
for expression in ["3 + 4", "10 - 2.5", "6 * 7", "10 / 4", "2 % 3"]:
    print(expression)
    final = graph.invoke({"expression": expression})
    print("  answer:", final["answer"])

3 + 4
  -> add
  answer: 3 + 4 = 7
10 - 2.5
  -> subtract
  answer: 10 - 2.5 = 7.5
6 * 7
  -> multiply
  answer: 6 * 7 = 42
10 / 4
  -> divide
  answer: 10 / 4 = 2.5
2 % 3
  -> error
  answer: Sorry, I don't know the operator '%'. Use +, -, * or /.


And the full final state for one run of each kind. Notice that the `error` run has no `result` key, because no
node ever wrote one.

In [9]:
print(graph.invoke({"expression": "6 * 7"}))
print(graph.invoke({"expression": "2 % 3"}))

  -> multiply
{'expression': '6 * 7', 'a': 6.0, 'op': '*', 'b': 7.0, 'result': 42.0, 'answer': '6 * 7 = 42'}
  -> error
{'expression': '2 % 3', 'a': 2.0, 'op': '%', 'b': 3.0, 'answer': "Sorry, I don't know the operator '%'. Use +, -, * or /."}


## Step 8: Why the `Literal` hint matters

To see why, here is the same router **without** the return type. We need to build the graph again, so we put the
wiring from Step 5 in a function, `build_calculator`. It's the same code with the router and `path_map` (Step 9) as
inputs.

In [10]:
def build_calculator(router, path_map=None):
    builder = StateGraph(CalcState)
    builder.add_node("parse", parse)
    for name, fn in [("add", add), ("subtract", subtract), ("multiply", multiply), ("divide", divide)]:
        builder.add_node(name, fn)
        builder.add_edge(name, "format")
    builder.add_node("format", format_answer)
    builder.add_node("error", error)
    builder.add_edge(START, "parse")
    builder.add_conditional_edges("parse", router, path_map)
    builder.add_edge("format", END)
    builder.add_edge("error", END)
    return builder.compile()

`route_no_hint` returns exactly the same names. The graph still runs correctly. But LangGraph can no longer see
which nodes the router might pick, so the diagram is **wrong**: it shows a single arrow `parse --> __end__` and no
arrows into the operation nodes at all.

In [11]:
def route_no_hint(state):  # same logic, no Literal return type
    return OPERATIONS.get(state["op"], "error")


unhinted = build_calculator(route_no_hint)
print(unhinted.invoke({"expression": "3 + 4"})["answer"])
print(unhinted.get_graph().draw_mermaid())

  -> add
3 + 4 = 7
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__(<p>__start__</p>)
	parse(parse)
	add(add)
	subtract(subtract)
	multiply(multiply)
	divide(divide)
	format(format)
	error(error)
	__end__(<p>__end__</p>)
	__start__ --> parse;
	parse --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 9: The explicit `path_map` form

Instead of a `Literal` hint, you can give `add_conditional_edges` a third argument, a **`path_map`**. It's a dict from
whatever the router returns to the name of a node. This lets the router return short labels (here the operator
symbol itself) while the graph decides which node each label means. As a bonus, the labels appear on the arrows in
the diagram.

In [12]:
def route_by_symbol(state: CalcState) -> str:
    op = state["op"]
    return op if op in OPERATIONS else "unknown"


path_map = {"+": "add", "-": "subtract", "*": "multiply", "/": "divide", "unknown": "error"}
mapped = build_calculator(route_by_symbol, path_map)
print(mapped.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	add(add)
	subtract(subtract)
	multiply(multiply)
	divide(divide)
	format(format)
	error(error)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	add --> format;
	divide --> format;
	multiply --> format;
	parse -. &nbsp;+&nbsp; .-> add;
	parse -. &nbsp;/&nbsp; .-> divide;
	parse -. &nbsp;unknown&nbsp; .-> error;
	parse -. &nbsp;*&nbsp; .-> multiply;
	parse -. &nbsp;-&nbsp; .-> subtract;
	subtract --> format;
	error --> __end__;
	format --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



The routing works just as before. The `path_map` only changes how the router's answers become node names.

In [13]:
for expression in ["3 + 4", "6 * 7", "2 % 3"]:
    print(expression, "->", mapped.invoke({"expression": expression})["answer"])

  -> add
3 + 4 -> 3 + 4 = 7
  -> multiply
6 * 7 -> 6 * 7 = 42
  -> error
2 % 3 -> Sorry, I don't know the operator '%'. Use +, -, * or /.


**When to use which?** A `Literal` hint is shorter and keeps everything in the router. A `path_map` is handy when
the router's natural answer isn't a node name (a symbol, a `True`/`False`, a category). It also puts labels on the
diagram. Both let LangGraph check the routes and draw them. Use one of them every time.

## What just happened

- **Step 4** tested the router on its own: `'+'` routes to `'add'`, and `'%'` routes to `'error'`.
- **Steps 5–6** replaced the fixed edge after `parse` with `add_conditional_edges("parse", route)`. The diagram shows
  dashed arrows `parse -.-> add`, `subtract`, `multiply`, `divide` and `error`.
- **Step 7** ran five expressions. Each printed exactly one `-> name` line, showing the single route taken. `2 % 3`
  went to `error` and its final state has no `result`.
- **Step 8** removed the `Literal` hint. The answers were still right, but the diagram wrongly showed
  `parse --> __end__`.
- **Step 9** used a `path_map` instead. The router returned symbols such as `'+'`, and the diagram labelled each
  dashed arrow with them.

## Common mistakes

1. **The router returns a name that isn't listed.** With a `path_map`, or a `Literal` hint, a value that isn't
   listed fails when the graph runs, for example:
   `KeyError: '%'`.
   Fix: make sure every possible return value is listed, including a fallback such as `"error"`.
2. **No `Literal` hint and no `path_map`.** There's **no error**, but the diagram is wrong (Step 8). Worse, if
   such a router returns a name with no node, LangGraph only logs a warning (`wrote to unknown channel ...,
   ignoring it`) and the graph quietly stops.
   Fix: always give the router a `Literal[...]` return type or pass a `path_map`.
3. **A `path_map` points at a node that doesn't exist.** `compile()` catches it:
   `ValueError: At 'parse' node, 'route_by_symbol' branch found unknown target 'divde'`.
   Fix: correct the node name in the `path_map`.

## Try it

1. `10 / 0` still crashes inside `divide` with `ZeroDivisionError`. Add a node `divide_by_zero` that sets
   `answer` to `"10 / 0: can't divide by zero"` (using the real expression), and change the router so that `/` with
   `b == 0` goes there. Remember to update the `Literal` hint and wire the new node to `END`.
   (Lesson 14 will instead *pause and ask a human* for a new divisor.)
2. Add a `%` (remainder) operation: a `modulo` node, a new entry in `OPERATIONS`, and the name in the `Literal`.
   Check that `"17 % 5"` gives `17 % 5 = 2`, and print the diagram.

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **Conditional edge**: an edge whose destination is chosen while the graph runs, by a router function.
- **Router function**: a function that reads the state and returns the name of the next node. It doesn't change the
  state.
- **`add_conditional_edges`**: registers a router to run after a given node.
- **`Literal[...]`**: a type hint meaning "exactly one of these values". LangGraph reads it to learn a router's
  possible destinations.
- **`path_map`**: a dict from a router's return values to node names, passed to `add_conditional_edges`.